# Decoder-only Transformer trained on FineWeb-Edu
## Baseline configuration with GPT-2 BPE tokenizer

In [76]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
import time
import gc
from transformers import AutoTokenizer
from datasets import load_dataset

In [77]:
device = torch.device("cuda")
print(device)
if torch.cuda.is_available():
    print('cuda available with GPU:',torch.cuda.get_device_name(0))

cuda
cuda available with GPU: NVIDIA GeForce RTX 5080


## Tokenizer
Swap this block to test different tokenizers.  
The rest of the pipeline reads `tokenizer.vocab_size` and `tokenizer.eos_token_id`.

In [78]:
# --- Tokenizer (swap this block for ablation experiments) ---
tokenizer = AutoTokenizer.from_pretrained("gpt2")
vocab_size = tokenizer.vocab_size  # 50257
eos_token_id = tokenizer.eos_token_id  # 50256
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {vocab_size}")
print(f"EOS token id: {eos_token_id}")

Tokenizer: GPT2Tokenizer
Vocabulary size: 50257
EOS token id: 50256


## Data pipeline
Stream FineWeb-Edu from HuggingFace. Documents are tokenized, separated by EOS, and packed into `(seq_length, batch_size)` chunks — the same tensor shape the model expects.

In [79]:
def streaming_token_batcher(dataset_iter, tokenizer, batch_size, seq_length):
    """
    Yields (data, target) tuples of shape (seq_length, batch_size).
    Concatenates tokenized documents (separated by EOS) into a flat buffer,
    then carves out chunks matching the tensor contract the model expects.
    """
    buffer = []
    chunk_size = batch_size * (seq_length + 1)  # +1 for the target offset

    for example in dataset_iter:
        tokens = tokenizer.encode(example["text"])
        tokens.append(eos_token_id)
        buffer.extend(tokens)

        while len(buffer) >= chunk_size:
            chunk = torch.LongTensor(buffer[:chunk_size])
            buffer = buffer[chunk_size:]

            chunk = chunk.view(batch_size, seq_length + 1)
            data   = chunk[:, :seq_length].t().contiguous()    # (seq_length, batch_size)
            target = chunk[:, 1:seq_length+1].t().contiguous() # (seq_length, batch_size)
            yield data, target


def get_epoch_batches(tokens_per_epoch, batch_size, seq_length, tokenizer,
                      dataset_name="HuggingFaceFW/fineweb-edu",
                      subset="sample-10BT", split="train", seed=0):
    """Stream one epoch's worth of training batches."""
    ds = load_dataset(dataset_name, name=subset, split=split, streaming=True)
    ds = ds.shuffle(seed=seed)
    batcher = streaming_token_batcher(ds, tokenizer, batch_size, seq_length)

    batches_per_epoch = tokens_per_epoch // (batch_size * seq_length)
    try:
        for i, (data, target) in enumerate(batcher):
            if i >= batches_per_epoch:
                break
            yield data, target
    finally:
        batcher.close()
        del ds
        gc.collect()


def load_eval_buffer(tokenizer, batch_size, num_tokens=100_000,
                     dataset_name="HuggingFaceFW/fineweb-edu",
                     subset="sample-10BT", split="train"):
    """Pre-load a small eval buffer from a different region of the stream."""
    ds = load_dataset(dataset_name, name=subset, split=split, streaming=True)
    # skip ahead to avoid overlap with training data
    ds = ds.skip(50_000)
    buffer = []
    for example in ds:
        tokens = tokenizer.encode(example["text"])
        tokens.append(eos_token_id)
        buffer.extend(tokens)
        if len(buffer) >= num_tokens:
            break
    buffer = buffer[:num_tokens]
    del ds
    gc.collect()
    data = torch.LongTensor(buffer)
    # batchify: same layout as PTB
    nbatch = data.size(0) // batch_size
    data = data.narrow(0, 0, nbatch * batch_size)
    data = data.view(batch_size, -1).t().contiguous()
    return data

## Model architecture
Identical to the template transformer — sinusoidal PE, pre-norm decoder blocks, ReLU MLP.

In [80]:
# sinusoidal PE
def generate_positional_encoding(seq_length, dim):
    assert dim == 2* (dim//2) # check if dim is divisible by 2
    pe = torch.zeros(seq_length, dim)
    position = torch.arange(0, seq_length, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / dim))
    pe[:,0::2] = torch.sin(position * div_term)
    pe[:,1::2] = torch.cos(position * div_term)
    return pe

In [81]:
# One attention head
class AttentionHead(nn.Module):
    def __init__(self, d, d_head, dropout):
        super().__init__()
        self.LN_MHA = nn.LayerNorm(d_head)
        self.LN_MLP = nn.LayerNorm(d_head)
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.dropout = nn.Dropout(dropout)
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); batch_len = H.size(1)
        # Compute a single attention head H = Softmax( QK^T / d^0.5 ) V        
        Q = self.query(H) # size=[batch_size, batch_length, d_head]        
        K = self.key(H) # size=[batch_size, batch_length, d_head]
        V = self.value(H) # size=[batch_size, batch_length, d_head]
        attention_score = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(2)) # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        mask = torch.tril(torch.ones(batch_len,batch_len)).long().to(device) # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        attention_score = attention_score.masked_fill(mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=-1) # softmax over rows. sum weights = 1, size=[batch_size, batch_length, batch_len)
        attention_score = self.dropout(attention_score) # dropout attention scores
        H_HA = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        return H_HA # return prediction scores for next token

In [82]:
# One MHA block
class MultipleAttentionHead(nn.Module):
    def __init__(self, d, num_heads, dropout):
        super().__init__()
        d_head = d // num_heads # dim_head = d // num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ AttentionHead(d, d_head, dropout) for _ in range(num_heads) ])
        self.WO = nn.Linear(d, d) # combination layer
        self.dropout = nn.Dropout(dropout)
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); seq_length = H.size(1)
        H_heads = [] 
        H_heads = [self.MHA[i](H) for i in range(len(self.MHA))]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, seq_length, d]            
        H_heads = self.dropout(H_heads) # dropout attention activations
        H = self.WO(H_heads) # size=[batch_size, seq_length, d]
        return H

In [83]:
# One transformer block
class TransformerBlock(nn.Module):
    def __init__(self, d, num_heads, dropout):
        super().__init__()
        self.LN_MHA = nn.LayerNorm(d)
        self.LN_MLP = nn.LayerNorm(d)
        self.MHA = MultipleAttentionHead(d, num_heads, dropout)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4*d,d))        
    def forward(self, H): # size=[batch_size, seq_length, d]
        # Multiple Attention Heads w/ layer normalization (LN), residual connection (RC) 
        H = H + self.MHA(self.LN_MHA(H))
        # MLP w/ layer normalization (LN), residual connection (RC) 
        H = H + self.MLP(self.LN_MLP(H))
        return H # size=[batch_size, seq_length, d]

In [84]:
# Transformer decoder before MLP
class Transformer_decoder(nn.Module):
    def __init__(self, d, num_heads, num_blocks, seq_length, dropout):
        super().__init__()
        self.TR_Blocks = nn.ModuleList([ TransformerBlock(d, num_heads, dropout) for _ in range(num_blocks) ]) 
    def forward(self, batch_seq, pos_enc):
        H = batch_seq.transpose(1,0) # size=[batch_size, seq_length, d]
        batch_size = H.size(0); batch_len = H.size(1)
        # Add positional encoding  
        pos_enc = pos_enc.unsqueeze(dim=0) # size=[1,          seq_length, d]
        H = H + pos_enc                     # size=[batch_size, seq_length, d]
        # Apply transformer blocks 
        for TR_Block in self.TR_Blocks:
            H = TR_Block(H)
        # Output
        H = H.permute(1,0,2)  # size=[batch_length, batch_size, d]
        return H # return prediction scores for next token

In [ ]:
# End to end decoder only transformer (naive, without any funny tricks)
class ANN(nn.Module):
    
    def __init__(self, d, num_heads, num_blocks, seq_length, dropout):
        super(ANN, self).__init__()
        self.decoder = Transformer_decoder(d, num_heads, num_blocks, seq_length, dropout)
    
    def forward(self, g_seq , pos ):
        h_dec_seq = self.decoder( g_seq , pos )
        return h_dec_seq 
    

class attention_net(nn.Module):

    def __init__(self, d, num_heads, num_blocks, seq_length, dropout):
        super(attention_net, self).__init__()  
        self.layer1 = nn.Embedding(vocab_size, hidden_size)
        self.layer2 = ANN(d, num_heads, num_blocks, seq_length, dropout)
        self.layer3 = nn.Linear(hidden_size, vocab_size)
        self.layer3.weight = self.layer1.weight
        nn.init.normal_(self.layer1.weight, mean=0, std=0.02) # hacky hack, dont mind this :eyes:


    def forward(self, word_seq, pos ):
        g_seq     =   self.layer1( word_seq ) # size=(seq_length, bs, hidden_dim) 
        h_seq     =   self.layer2( g_seq , pos ) # size=(seq_length, bs, hidden_dim) 
        score_seq =   self.layer3( h_seq ) # size=(seq_length, bs, vocab_size)
        return score_seq 

In [86]:
def display_num_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    print('There are {} ({:.2f} million) parameters in this neural network'.format(
        nb_param, nb_param/1e6)
         )

## Hyperparameters & model instantiation

In [87]:
# hyperparameters
bs = 20
hidden_size = 128
num_heads = 4
num_blocks = 2
dropout = 0.1
seq_length = 100
tokens_per_epoch = 1_000_000  # how many tokens per training epoch

pos = generate_positional_encoding(seq_length, hidden_size).to(device)

net = attention_net(hidden_size, num_heads, num_blocks, seq_length, dropout)
print(net)
display_num_param(net)
net = net.to(device)

attention_net(
  (layer1): Embedding(50257, 128)
  (layer2): ANN(
    (decoder): Transformer_decoder(
      (TR_Blocks): ModuleList(
        (0-1): 2 x TransformerBlock(
          (LN_MHA): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (LN_MLP): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (MHA): MultipleAttentionHead(
            (MHA): ModuleList(
              (0-3): 4 x AttentionHead(
                (LN_MHA): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
                (LN_MLP): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
                (query): Linear(in_features=128, out_features=32, bias=False)
                (key): Linear(in_features=128, out_features=32, bias=False)
                (value): Linear(in_features=128, out_features=32, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
            )
            (WO): Linear(in_features=128, out_features=128, bias=True)
            (dropout): D

In [88]:
criterion = nn.CrossEntropyLoss()

my_lr = 0.001
optimizer = torch.optim.Adam(net.parameters(), lr=my_lr)

In [89]:
# pre-load eval buffer (streams ~100K tokens from a different region of FineWeb-Edu)
print("Loading eval buffer...")
eval_data = load_eval_buffer(tokenizer, bs)
print(f"Eval buffer shape: {eval_data.shape}")

Loading eval buffer...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1368 > 1024). Running this sequence through the model will result in indexing errors


Eval buffer shape: torch.Size([5000, 20])


In [90]:
# eval helper
def eval_on_test_set():
    net.eval()
    running_loss = 0
    num_batches = 0
    num_steps = eval_data.size(0) - seq_length

    for count in range(0, num_steps, seq_length):
        minibatch_data  = eval_data[count   : count+seq_length]
        minibatch_label = eval_data[count+1 : count+seq_length+1]
        # pos = generate_positional_encoding(seq_length, hidden_size)

        minibatch_data  = minibatch_data.to(device)
        minibatch_label = minibatch_label.to(device)
        # pos = pos.to(device)

        scores = net(minibatch_data, pos)

        minibatch_label = minibatch_label.view(bs * seq_length)
        scores = scores.view(bs * seq_length, vocab_size)

        loss = criterion(scores, minibatch_label)

        running_loss += loss.item()
        num_batches += 1

    total_loss = running_loss / num_batches
    print(f'eval: exp(loss) = {math.exp(total_loss):.2f}')
    net.train()

## Training

In [91]:
# training loop
start = time.time()
for epoch in range(20):

    # decay learning rate after epoch 2
    if epoch >= 2:
        optimizer.param_groups[0]['lr'] /= 1.1
        my_lr = optimizer.param_groups[0]['lr']

    running_loss = 0
    num_batches = 0
    net.train()

    for minibatch_data, minibatch_label in get_epoch_batches(
            tokens_per_epoch, bs, seq_length, tokenizer, seed=epoch):

        optimizer.zero_grad()

        # pos = generate_positional_encoding(seq_length, hidden_size)

        minibatch_data  = minibatch_data.to(device)
        minibatch_label = minibatch_label.to(device)
        # pos = pos.to(device)

        # forward pass: size=(seq_length, bs, vocab_size)
        scores = net(minibatch_data, pos)

        # reshape for cross-entropy
        scores = scores.view(bs * seq_length, vocab_size)
        minibatch_label = minibatch_label.view(bs * seq_length)

        loss = criterion(scores, minibatch_label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    total_loss = running_loss / num_batches
    elapsed = time.time() - start

    print(f'\nepoch={epoch}\t time={elapsed:.1f}\t lr={my_lr}\t exp(loss)={math.exp(total_loss):.2f}')
    eval_on_test_set()

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


epoch=0	 time=15.5	 lr=0.001	 exp(loss)=2438.83
eval: exp(loss) = 1981.61


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


epoch=1	 time=27.7	 lr=0.001	 exp(loss)=1374.21
eval: exp(loss) = 1582.72


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

...


epoch=17	 time=241.2	 lr=0.00021762913579014855	 exp(loss)=364.56
eval: exp(loss) = 425.54


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


epoch=18	 time=253.1	 lr=0.00019784466890013504	 exp(loss)=383.83
eval: exp(loss) = 420.63


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


epoch=19	 time=268.5	 lr=0.00017985878990921367	 exp(loss)=358.02
eval: exp(loss) = 404.57


## Text generation
Autoregressive generation: feed the prompt, sample one token, append it, repeat.  
Supports **greedy**, **top-k**, and **top-p (nucleus)** decoding with temperature scaling.

In [92]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=1.0, top_k=0, top_p=0.0):
    """
    Autoregressive text generation from a prompt string.
    
    Args:
        prompt:          seed text
        max_new_tokens:  how many tokens to generate
        temperature:     >1 = more random, <1 = more focused, 1 = unchanged
        top_k:           if >0, only sample from the top-k most likely tokens
        top_p:           if >0, nucleus sampling — sample from smallest set with cumulative prob >= top_p
    """
    net.eval()
    token_ids = tokenizer.encode(prompt)

    for _ in range(max_new_tokens):
        # truncate context to seq_length if it grows too long
        context = token_ids[-seq_length:]
        x = torch.LongTensor(context).unsqueeze(1).to(device)  # (ctx_len, 1)
        pos = generate_positional_encoding(x.size(0), hidden_size).to(device)

        # forward pass with batch_size=1
        scores = net(x, pos)       # (ctx_len, 1, vocab_size)
        logits = scores[-1, 0, :]  # last position logits, (vocab_size,)

        # temperature scaling
        if temperature != 1.0:
            logits = logits / temperature

        # top-k filtering
        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[-1]] = float('-inf')

        # top-p (nucleus) filtering
        if top_p > 0.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=0), dim=0)
            # remove tokens with cumulative prob above the threshold (keep first token above)
            remove_mask = cumulative_probs - F.softmax(sorted_logits, dim=0) >= top_p
            sorted_logits[remove_mask] = float('-inf')
            logits = sorted_logits.scatter(0, sorted_idx, sorted_logits)

        probs = F.softmax(logits, dim=0)

        # greedy vs sampling
        if temperature == 0 or (top_k == 0 and top_p == 0.0):
            next_token = torch.argmax(probs).item()
        else:
            next_token = torch.multinomial(probs, 1).item()

        # stop on EOS
        if next_token == eos_token_id:
            break

        token_ids.append(next_token)

    return tokenizer.decode(token_ids)

In [94]:
# sample prompts
prompts = [
    "The most important thing about education is",
    "Scientists recently discovered that",
    "In the history of mathematics, the concept of",
]

for p in prompts:
    print("=" * 70)
    print(f"PROMPT: {p}")
    print("-" * 70)

    print("\n[Greedy]")
    print(generate(p, max_new_tokens=100))

    print("\n[Top-k=50, temp=0.8]")
    print(generate(p, max_new_tokens=100, temperature=0.8, top_k=50))

    print("\n[Nucleus p=0.9, temp=0.9]")
    print(generate(p, max_new_tokens=100, temperature=0.9, top_p=0.9))
    print()

PROMPT: The most important thing about education is
----------------------------------------------------------------------

[Greedy]
The most important thing about education is a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be a good way to be

[Top-k=50, temp=0.8]
The most important thing about education is to achieve the right.
- As that the world we can be considered without the current as an additional market, so many of the early-century world and the number of the country. The state of government became a few weeks ago.
This year has a whole study has been published to the world and they think that what people have to make a new way to be aware of the year that are not a population of the world.
- There is a num